# Hướng dẫn chạy dự án trên Google Colab

Notebook này giúp bạn chạy pipeline `IEEECS_CPS_2026` trên Colab, đọc dữ liệu từ **Google Drive**.

## Chuẩn bị trên Google Drive

Chọn **một** trong hai cách sau:

**Cách 1 (khuyến nghị):** Đặt 8 file CSV CICIDS2017 vào:
```
MyDrive/IEEECS_CPS_2026/paper_1/data/raw/
```

**Cách 2:** Giữ CSV ở thư mục riêng trên Drive, ví dụ:
```
MyDrive/ids-2017/
```
Sau đó truyền `--input-dir` khi chạy `data_preparation.py`.

> Lưu ý: Colab **không cần Java/Spark**. Pipeline xử lý dữ liệu dùng pandas.

## Bước 1: Kết nối Google Drive

Upload thư mục dự án `IEEECS_CPS_2026` lên Drive, rồi mount Drive trong Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Bước 2: Vào thư mục dự án và kiểm tra dữ liệu

Sửa `PROJECT_DIR` nếu bạn lưu repo ở vị trí khác trên Drive.
Sửa `RAW_DATA_DIR` nếu CSV nằm ngoài `paper_1/data/raw/`.

In [ ]:
import os
from pathlib import Path

# Sửa 2 đường dẫn này cho khớp với Drive của bạn
PROJECT_DIR = "/content/drive/MyDrive/IEEECS_CPS_2026/paper_1"
RAW_DATA_DIR = f"{PROJECT_DIR}/data/raw"  # hoặc: "/content/drive/MyDrive/ids-2017"

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
print("Raw CSV directory:", RAW_DATA_DIR)
print("CSV files found:", len(list(Path(RAW_DATA_DIR).glob("*.csv"))))
!ls -la

## Bước 3: Cài đặt các thư viện cần thiết
Cài đặt phiên bản chính xác của các thư viện được yêu cầu trong `requirements.txt`.

In [ ]:
!pip install -r requirements.txt

## Bước 4: Chuẩn bị dữ liệu từ Google Drive

Script sẽ đọc CSV từ `RAW_DATA_DIR`, gộp/làm sạch, rồi lưu Parquet vào `paper_1/data/`.

In [ ]:
# Đọc CSV từ Google Drive (RAW_DATA_DIR đã khai báo ở bước 2)
!python code/data_preparation.py --input-dir {RAW_DATA_DIR}

# Nếu CSV ở thư mục khác, sửa trực tiếp:
# !python code/data_preparation.py --input-dir "/content/drive/MyDrive/ids-2017"

## Bước 5: Huấn luyện mô hình

Sau khi có `data/train_data.parquet` và `data/test_data.parquet`, chạy huấn luyện teacher và student.

In [ ]:
!python code/hybrid_ids_cicids2017.py

In [ ]:
!python code/knowledge_distillation.py

## Bước 6: Cập nhật hình ảnh và bảng LaTeX

Sau khi huấn luyện xong, các script dưới đây tạo biểu đồ và cập nhật số liệu vào `sections/results.tex`.
Kết quả được lưu trên Drive nên bạn có thể tải về hoặc mở trực tiếp từ Drive.

In [ ]:
!python code/replot_figures.py
!python code/populate_latex_tables.py